# Car Detector — Entrenar YOLOv8 patentes (GPU Colab)\n\n1. Runtime → Change runtime type → **GPU**\n2. Subí `plate_dataset.zip` (generado con `scripts/pack_plate_dataset.py`)\n3. Ejecutá todas las celdas\n4. Descargá `best.pt`

In [ ]:
!nvidia-smi\nimport torch\nprint('CUDA:', torch.cuda.is_available())\nprint('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip -q install ultralytics

In [ ]:
from google.colab import files\nimport zipfile\nfrom pathlib import Path\n\nuploaded = files.upload()  # elegí plate_dataset.zip\nzip_name = next(iter(uploaded))\nPath('dataset').mkdir(exist_ok=True)\nwith zipfile.ZipFile(zip_name, 'r') as zf:\n    zf.extractall('dataset')\nprint('OK', list(Path('dataset').iterdir()))

In [ ]:
from pathlib import Path\nimport random\n\nroot = Path('dataset')\nimages = sorted((root/'images').glob('*.*'))\nusable = [p for p in images if (root/'labels'/f'{p.stem}.txt').exists()]\nrandom.Random(42).shuffle(usable)\nn_val = max(1, int(len(usable)*0.2))\nval, train = usable[:n_val], usable[n_val:]\n\n(root/'train.txt').write_text('\\n'.join(str(p.resolve()) for p in train) + '\\n')\n(root/'val.txt').write_text('\\n'.join(str(p.resolve()) for p in val) + '\\n')\n\nyaml = f\"\"\"path: {root.resolve()}\ntrain: train.txt\nval: val.txt\nnames:\n  0: licence\nnc: 1\n\"\"\"\n(root/'data.yaml').write_text(yaml)\nprint(f'train={len(train)} val={len(val)}')

In [ ]:
from ultralytics import YOLO\n\nmodel = YOLO('yolov8n.pt')\nmodel.train(\n    data='dataset/data.yaml',\n    epochs=40,\n    imgsz=640,\n    batch=16,\n    device=0,  # GPU Colab\n    name='plate_yolo',\n    exist_ok=True,\n)\nprint('Listo')

In [ ]:
from google.colab import files\nfrom pathlib import Path\n\nbest = Path('runs/detect/plate_yolo/weights/best.pt')\nassert best.exists(), best\nfiles.download(str(best))